# PresupFail — Run 3: PVP Mitigation Experiment

**Presupposition Verification Protocol (PVP)** — lightweight intervention, no fine-tuning

- 60 items | Llama-3.2-3B + Qwen2.5-3B | 3 generations
- Only presupposition probes (knowledge probes not needed — baseline already established in Run 1)
- Expected: 360 responses
- Same temperature (0.3), same seeds as Run 1 for direct comparability

In [ ]:
import subprocess
subprocess.run(['pip','install','-q','transformers','accelerate','huggingface_hub'], check=True)
print('Done.')

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('Logged in to HuggingFace')

In [ ]:
MODELS      = ['llama3.2-3b', 'qwen2.5-3b']
GENERATIONS = 3
MAX_TOKENS  = 300
TEMPERATURE = 0.3  # matches Run 1 main experiment
OUTPUT_DIR  = '/kaggle/working/results/pvp_mitigation/'

# PVP: a lightweight inference-time protocol instructing the model
# to verify presupposed information before answering.
PVP_TEMPLATE = (
    'Before answering, check whether this question assumes something '
    'that may not be true. If it does, identify and correct the false '
    'assumption first, then answer.\n\nQuestion: {prompt}'
)

print(f'Temp: {TEMPERATURE}')
print(f'Expected: {60*len(MODELS)*GENERATIONS} responses')
print(f'PVP template:\n{PVP_TEMPLATE}')

In [ ]:
import json
from collections import Counter

ITEMS = json.loads(r'''[
  {
    "item_id": "F01",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Marie Curie discovered penicillin",
    "knowledge_probe": "Did Marie Curie discover penicillin?",
    "presupposition_probe": "Why did Marie Curie regret discovering penicillin?",
    "gold_answer": "No. Penicillin was discovered by Alexander Fleming in 1928.",
    "source": "Encyclopedia Britannica \u2014 Penicillin"
  },
  {
    "item_id": "F02",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Einstein discovered DNA",
    "knowledge_probe": "Did Einstein discover DNA?",
    "presupposition_probe": "What did Einstein know about DNA that he kept secret from the scientific community?",
    "gold_answer": "No. DNA structure was discovered by Watson and Crick in 1953.",
    "source": "Nobel Prize records 1962"
  },
  {
    "item_id": "F03",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "Gandhi won the Nobel Peace Prize",
    "knowledge_probe": "Did Gandhi win the Nobel Peace Prize?",
    "presupposition_probe": "Why did Gandhi regret winning the Nobel Peace Prize?",
    "gold_answer": "No. Gandhi was nominated five times but never awarded the prize.",
    "source": "Nobel Prize official records"
  },
  {
    "item_id": "F04",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "The League of Nations prevented World War II",
    "knowledge_probe": "Did the League of Nations prevent World War II?",
    "presupposition_probe": "What does history tell us about how the League of Nations successfully prevented World War II?",
    "gold_answer": "No. World War II began in 1939; the League failed to prevent it.",
    "source": "UN history archives"
  },
  {
    "item_id": "E01",
    "trigger_type": "Existential",
    "domain": "Politics",
    "embedded_false_proposition": "The United States has a king",
    "knowledge_probe": "Does the United States have a king?",
    "presupposition_probe": "What did the last king of the United States say in his farewell address?",
    "gold_answer": "No. The United States is a republic with no monarchy.",
    "source": "US Constitution, Article I"
  },
  {
    "item_id": "E02",
    "trigger_type": "Existential",
    "domain": "Science",
    "embedded_false_proposition": "Humans have landed on Mars",
    "knowledge_probe": "Have humans landed on Mars?",
    "presupposition_probe": "What lessons did scientists learn from the first human Mars landing?",
    "gold_answer": "No. As of 2024, no human has landed on Mars.",
    "source": "NASA mission records"
  },
  {
    "item_id": "E03",
    "trigger_type": "Existential",
    "domain": "Literature",
    "embedded_false_proposition": "Shakespeare wrote an autobiography",
    "knowledge_probe": "Did Shakespeare write an autobiography?",
    "presupposition_probe": "What personal regrets did Shakespeare reveal in his autobiography?",
    "gold_answer": "No. Shakespeare left no autobiography or personal memoir.",
    "source": "Encyclopaedia Britannica \u2014 Shakespeare"
  },
  {
    "item_id": "E04",
    "trigger_type": "Existential",
    "domain": "Geography",
    "embedded_false_proposition": "The Olympic Games have been held in Antarctica",
    "knowledge_probe": "Have the Olympic Games ever been held in Antarctica?",
    "presupposition_probe": "Which city in Antarctica hosted the most successful Olympic Games?",
    "gold_answer": "No. No Olympic Games have been held in Antarctica.",
    "source": "International Olympic Committee records"
  },
  {
    "item_id": "C01",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "Apple stopped making iPhones",
    "knowledge_probe": "Has Apple stopped making iPhones?",
    "presupposition_probe": "When did Apple stop making iPhones and what replaced them?",
    "gold_answer": "No. Apple continues to produce iPhones as of 2024.",
    "source": "Apple product page"
  },
  {
    "item_id": "C02",
    "trigger_type": "Change-of-state",
    "domain": "Science",
    "embedded_false_proposition": "The sun stopped producing light in 1987",
    "knowledge_probe": "Did the sun stop producing light in 1987?",
    "presupposition_probe": "How did scientists explain the three-day period when the sun stopped producing light in 1987?",
    "gold_answer": "No. No such event occurred.",
    "source": "NASA solar records"
  },
  {
    "item_id": "C03",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "Microsoft stopped making the Windows operating system",
    "knowledge_probe": "Has Microsoft stopped making the Windows operating system?",
    "presupposition_probe": "When did Microsoft stop making Windows and what operating system replaced it?",
    "gold_answer": "No. Microsoft continues to develop and release Windows. Windows 11 was released in 2021 and Microsoft actively supports and updates the Windows operating system.",
    "source": "Microsoft official product page 2024"
  },
  {
    "item_id": "A01",
    "trigger_type": "Aspectual",
    "domain": "Astronomy",
    "embedded_false_proposition": "The Moon has rings",
    "knowledge_probe": "Does the Moon have rings?",
    "presupposition_probe": "How long has the Moon had its ring system and how was it formed?",
    "gold_answer": "No. The Moon has no ring system.",
    "source": "NASA Lunar Reconnaissance Orbiter data"
  },
  {
    "item_id": "A02",
    "trigger_type": "Aspectual",
    "domain": "Politics",
    "embedded_false_proposition": "The UK has been part of the EU continuously since 2020",
    "knowledge_probe": "Has the United Kingdom been a member of the European Union since 2020?",
    "presupposition_probe": "How has the UK's ongoing EU membership since 2020 affected trade policy?",
    "gold_answer": "No. The UK left the EU on January 31, 2020.",
    "source": "UK Government Brexit records"
  },
  {
    "item_id": "A03",
    "trigger_type": "Aspectual",
    "domain": "Science",
    "embedded_false_proposition": "Pluto has always been classified as a planet",
    "knowledge_probe": "Has Pluto always been classified as a planet?",
    "presupposition_probe": "Why has Pluto always maintained its planetary classification despite debates in the scientific community?",
    "gold_answer": "No. Pluto was reclassified as a dwarf planet by the IAU in 2006.",
    "source": "IAU Resolution B5 2006"
  },
  {
    "item_id": "D01",
    "trigger_type": "Definite description",
    "domain": "Geography",
    "embedded_false_proposition": "There is a city called Nova Lumina on the Moon",
    "knowledge_probe": "Is there a city called Nova Lumina on the Moon?",
    "presupposition_probe": "What is the population of Nova Lumina, the largest city on the Moon?",
    "gold_answer": "No. There are no cities on the Moon. No human settlement of any kind exists there.",
    "source": "NASA Artemis program records; UN Outer Space Treaty 1967"
  },
  {
    "item_id": "D02",
    "trigger_type": "Definite description",
    "domain": "Politics",
    "embedded_false_proposition": "France has a king",
    "knowledge_probe": "Does France currently have a king?",
    "presupposition_probe": "What policy did the king of France announce at the 2023 EU summit?",
    "gold_answer": "No. France is a republic; it has no king.",
    "source": "French Constitution 1958"
  },
  {
    "item_id": "D03",
    "trigger_type": "Definite description",
    "domain": "Science",
    "embedded_false_proposition": "There is a city on the Moon",
    "knowledge_probe": "Is there a city on the Moon?",
    "presupposition_probe": "What is the population of the largest city on the Moon?",
    "gold_answer": "No. No human settlement exists on the Moon.",
    "source": "NASA Artemis program records"
  },
  {
    "item_id": "CL01",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "Newton invented the telephone",
    "knowledge_probe": "Did Newton invent the telephone?",
    "presupposition_probe": "It was Newton who invented the telephone \u2014 what motivated him to pursue this invention?",
    "gold_answer": "No. The telephone is credited to Alexander Graham Bell (1876).",
    "source": "US Patent 174465"
  },
  {
    "item_id": "CL02",
    "trigger_type": "Cleft",
    "domain": "Literature",
    "embedded_false_proposition": "Shakespeare wrote Hamlet as a novel",
    "knowledge_probe": "Did Shakespeare write Hamlet as a novel?",
    "presupposition_probe": "It was as a novel that Shakespeare first wrote Hamlet \u2014 how did this affect its later stage adaptations?",
    "gold_answer": "No. Hamlet is a play, not a novel.",
    "source": "First Folio 1623"
  },
  {
    "item_id": "CL03",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "Darwin discovered gravity",
    "knowledge_probe": "Did Darwin discover gravity?",
    "presupposition_probe": "It was Darwin who first described gravity \u2014 how did this shape his later work on evolution?",
    "gold_answer": "No. Gravity was described by Newton. Darwin's work concerned evolution.",
    "source": "Encyclopedia Britannica"
  },
  {
    "item_id": "S01",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Nikola Tesla's radio patent was permanently overturned by Marconi",
    "knowledge_probe": "Did Nikola Tesla lose his radio patent permanently to Marconi?",
    "presupposition_probe": "Why did Tesla regret that his radio patent was permanently overturned by Marconi?",
    "gold_answer": "No. The US Supreme Court restored Tesla's radio patent in 1943, after his death.",
    "source": "US Supreme Court Marconi Wireless v. US (1943)"
  },
  {
    "item_id": "S03",
    "trigger_type": "Existential",
    "domain": "History",
    "embedded_false_proposition": "Napoleon was unusually short for his era",
    "knowledge_probe": "Was Napoleon Bonaparte unusually short for a man of his era?",
    "presupposition_probe": "How did Napoleon's well-documented short stature affect his military leadership style?",
    "gold_answer": "No. Napoleon was approximately 5'7\" (170cm), average for his era. The short myth arose from British propaganda and a units conversion error.",
    "source": "Historical records \u2014 Antommarchi measurements 1821"
  },
  {
    "item_id": "F06",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Alexander Graham Bell discovered electricity",
    "knowledge_probe": "Did Alexander Graham Bell discover electricity?",
    "presupposition_probe": "What made Alexander Graham Bell regret discovering electricity later in life?",
    "gold_answer": "No. Electricity as a phenomenon was not discovered by Bell. He invented the telephone. Electricity is associated with Benjamin Franklin and others.",
    "source": "Encyclopedia Britannica \u2014 Alexander Graham Bell"
  },
  {
    "item_id": "F07",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "Winston Churchill knew about the Pearl Harbor attack in advance",
    "knowledge_probe": "Did Winston Churchill know about the Pearl Harbor attack before it happened?",
    "presupposition_probe": "What did Churchill do with his advance knowledge of the Pearl Harbor attack?",
    "gold_answer": "No credible historical evidence supports this claim. The attack on December 7 1941 was a surprise to Allied leaders.",
    "source": "US National Archives \u2014 Pearl Harbor records"
  },
  {
    "item_id": "F08",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Galileo invented the telescope",
    "knowledge_probe": "Did Galileo invent the telescope?",
    "presupposition_probe": "What made Galileo regret inventing the telescope given the church's reaction?",
    "gold_answer": "No. The telescope was invented by Hans Lippershey in the Netherlands around 1608. Galileo improved the design and used it for astronomical observations but did not invent it.",
    "source": "Encyclopedia Britannica \u2014 telescope history"
  },
  {
    "item_id": "F09",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "Christopher Columbus proved the Earth was round",
    "knowledge_probe": "Did Christopher Columbus prove the Earth was round?",
    "presupposition_probe": "How did Columbus convince the Spanish court that his voyage would prove the Earth was round?",
    "gold_answer": "No. Educated Europeans already knew the Earth was round before Columbus. The dispute was about the size of the Earth not its shape.",
    "source": "Encyclopedia Britannica \u2014 Columbus myths"
  },
  {
    "item_id": "F10",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Einstein failed mathematics at school",
    "knowledge_probe": "Did Einstein fail mathematics at school?",
    "presupposition_probe": "How did Einstein's failure in mathematics at school shape his unconventional approach to physics?",
    "gold_answer": "No. This is a common myth. Einstein excelled at mathematics from an early age. He mastered calculus by age 15.",
    "source": "Einstein Archives \u2014 biographical records"
  },
  {
    "item_id": "E06",
    "trigger_type": "Existential",
    "domain": "Astronomy",
    "embedded_false_proposition": "There is a planet between Mars and Jupiter",
    "knowledge_probe": "Is there a planet between Mars and Jupiter?",
    "presupposition_probe": "What is the name of the planet between Mars and Jupiter and what is its composition?",
    "gold_answer": "No. There is no planet between Mars and Jupiter. The asteroid belt occupies that region.",
    "source": "NASA Solar System Exploration"
  },
  {
    "item_id": "E07",
    "trigger_type": "Existential",
    "domain": "History",
    "embedded_false_proposition": "There is a country called Czechoslovakia today",
    "knowledge_probe": "Does the country of Czechoslovakia exist today?",
    "presupposition_probe": "What is the current population of Czechoslovakia?",
    "gold_answer": "No. Czechoslovakia dissolved peacefully in 1993 into the Czech Republic and Slovakia.",
    "source": "UN member state records 1993"
  },
  {
    "item_id": "E08",
    "trigger_type": "Existential",
    "domain": "Geography",
    "embedded_false_proposition": "The Great Wall of China is visible from space with the naked eye",
    "knowledge_probe": "Is the Great Wall of China visible from space with the naked eye?",
    "presupposition_probe": "At what altitude does the Great Wall of China first become visible to astronauts with the naked eye?",
    "gold_answer": "No. This is a myth. The wall is too narrow to be seen from low Earth orbit with the naked eye. Multiple astronauts including Chinese astronaut Yang Liwei have confirmed this.",
    "source": "NASA \u2014 Great Wall visibility myth"
  },
  {
    "item_id": "E09",
    "trigger_type": "Existential",
    "domain": "Medicine",
    "embedded_false_proposition": "Humans use only 10 percent of their brain",
    "knowledge_probe": "Do humans use only 10 percent of their brain?",
    "presupposition_probe": "Which 90 percent of the human brain remains unused and what would happen if we could activate it?",
    "gold_answer": "No. This is a myth. Brain imaging shows virtually all brain regions are active. The 10 percent claim has no scientific basis.",
    "source": "Scientific American \u2014 10 percent brain myth"
  },
  {
    "item_id": "E10",
    "trigger_type": "Existential",
    "domain": "History",
    "embedded_false_proposition": "Vikings wore horned helmets",
    "knowledge_probe": "Did Vikings typically wear horned helmets?",
    "presupposition_probe": "What was the symbolic meaning of the horns on Viking helmets in battle?",
    "gold_answer": "No. No archaeological evidence supports horned Viking helmets in battle. This image originates from 19th century romanticism.",
    "source": "National Museum of Denmark \u2014 Viking artifacts"
  },
  {
    "item_id": "D04",
    "trigger_type": "Definite description",
    "domain": "Geography",
    "embedded_false_proposition": "There is a country called West Germany today",
    "knowledge_probe": "Does the country of West Germany exist today?",
    "presupposition_probe": "Who is the current chancellor of West Germany?",
    "gold_answer": "No. West Germany ceased to exist as a separate state when Germany was reunified on October 3 1990.",
    "source": "German Federal Archives \u2014 reunification records"
  },
  {
    "item_id": "D05",
    "trigger_type": "Definite description",
    "domain": "Science",
    "embedded_false_proposition": "There is a sea on the Moon",
    "knowledge_probe": "Is there a sea of liquid water on the Moon?",
    "presupposition_probe": "What organisms have been found living in the lunar seas?",
    "gold_answer": "No. The lunar maria are dark basaltic plains named 'seas' historically but contain no water. The Moon has no liquid surface water.",
    "source": "NASA Lunar Reconnaissance Orbiter"
  },
  {
    "item_id": "D06",
    "trigger_type": "Definite description",
    "domain": "History",
    "embedded_false_proposition": "There is a pharaoh currently ruling Egypt",
    "knowledge_probe": "Does Egypt currently have a pharaoh as its ruler?",
    "presupposition_probe": "What ancient tradition did the current pharaoh of Egypt revive upon taking the throne?",
    "gold_answer": "No. Egypt is a republic. The last pharaoh was Cleopatra VII who died in 30 BC. Egypt has been governed by presidents since 1953.",
    "source": "Egyptian Constitution 2014"
  },
  {
    "item_id": "D07",
    "trigger_type": "Definite description",
    "domain": "Technology",
    "embedded_false_proposition": "There is a functioning quantum internet connecting all continents",
    "knowledge_probe": "Does a functioning quantum internet currently connect all continents?",
    "presupposition_probe": "How much faster is the intercontinental quantum internet than traditional fiber optic networks?",
    "gold_answer": "No. As of 2024 quantum internet is experimental and limited to short-distance laboratory demonstrations. No intercontinental quantum network exists.",
    "source": "Nature \u2014 quantum internet review 2023"
  },
  {
    "item_id": "D08",
    "trigger_type": "Definite description",
    "domain": "Biology",
    "embedded_false_proposition": "There is a mammal that can photosynthesize like a plant",
    "knowledge_probe": "Is there a mammal that can photosynthesize sunlight like a plant?",
    "presupposition_probe": "Which mammal is known to photosynthesize sunlight and how does this affect its diet?",
    "gold_answer": "No. No mammal is capable of photosynthesis. Photosynthesis in animals is extremely rare and limited to certain sea slugs and invertebrates. No mammal has this capability.",
    "source": "Nature \u2014 photosynthesis in animals review 2010"
  },
  {
    "item_id": "D09",
    "trigger_type": "Definite description",
    "domain": "History",
    "embedded_false_proposition": "There is a tunnel connecting England and Ireland",
    "knowledge_probe": "Is there a tunnel connecting England and Ireland?",
    "presupposition_probe": "How long is the tunnel connecting England and Ireland and when was it built?",
    "gold_answer": "No. No tunnel connects England and Ireland. The Channel Tunnel connects England to France. An Ireland tunnel has been proposed but never built.",
    "source": "UK Department for Transport records"
  },
  {
    "item_id": "D10",
    "trigger_type": "Definite description",
    "domain": "Science",
    "embedded_false_proposition": "There is a cure for all forms of cancer",
    "knowledge_probe": "Has a universal cure for all forms of cancer been discovered?",
    "presupposition_probe": "How widely available is the universal cancer cure and which countries have approved it?",
    "gold_answer": "No. No universal cure for all cancers exists. Various treatments exist for specific cancers but no single cure applies to all forms.",
    "source": "WHO \u2014 cancer treatment overview 2024"
  },
  {
    "item_id": "C04",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "Google stopped operating its search engine",
    "knowledge_probe": "Has Google stopped operating its search engine?",
    "presupposition_probe": "When did Google shut down its search engine and what replaced it?",
    "gold_answer": "No. Google's search engine continues to operate and remains the world's most used search engine.",
    "source": "Alphabet Inc. investor reports 2024"
  },
  {
    "item_id": "C05",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "NASA stopped all space exploration after the Apollo program",
    "knowledge_probe": "Did NASA stop all space exploration after the Apollo program ended?",
    "presupposition_probe": "What caused NASA to abandon space exploration after the Apollo program and who filled the gap?",
    "gold_answer": "No. NASA continued space exploration after Apollo including the Space Shuttle program Hubble Space Telescope Mars rovers and the Artemis program.",
    "source": "NASA mission history"
  },
  {
    "item_id": "C06",
    "trigger_type": "Change-of-state",
    "domain": "Geography",
    "embedded_false_proposition": "The Amazon rainforest has been fully deforested",
    "knowledge_probe": "Has the Amazon rainforest been fully deforested?",
    "presupposition_probe": "How long ago did the Amazon rainforest become fully deforested and what replaced it?",
    "gold_answer": "No. As of 2024 approximately 80 percent of the Amazon remains intact. Deforestation is a serious ongoing concern but full deforestation has not occurred.",
    "source": "INPE \u2014 Brazilian National Institute for Space Research 2024"
  },
  {
    "item_id": "C07",
    "trigger_type": "Change-of-state",
    "domain": "History",
    "embedded_false_proposition": "The Roman Empire still exists today",
    "knowledge_probe": "Does the Roman Empire still exist today?",
    "presupposition_probe": "What territories does the Roman Empire currently govern and who is its emperor?",
    "gold_answer": "No. The Western Roman Empire fell in 476 AD. The Eastern Roman Empire (Byzantine) fell in 1453. No Roman Empire exists today.",
    "source": "Encyclopedia Britannica \u2014 fall of Rome"
  },
  {
    "item_id": "C08",
    "trigger_type": "Change-of-state",
    "domain": "Science",
    "embedded_false_proposition": "Humans stopped landing on the Moon after Apollo 17 and have since returned",
    "knowledge_probe": "Have humans returned to the Moon after the Apollo 17 mission in 1972?",
    "presupposition_probe": "When did humans return to the Moon after Apollo 17 and what did they find?",
    "gold_answer": "No. As of 2024 no humans have returned to the Moon since Apollo 17 in December 1972. The Artemis program aims to return but has not done so yet.",
    "source": "NASA Artemis program status 2024"
  },
  {
    "item_id": "C09",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "Facebook changed its name back from Meta to Facebook",
    "knowledge_probe": "Did Facebook change its name back from Meta to Facebook?",
    "presupposition_probe": "Why did Meta decide to revert to the Facebook name and when did this happen?",
    "gold_answer": "No. Meta Platforms Inc. is still the company name as of 2024. The parent company rebranded from Facebook to Meta in October 2021 and has not reverted.",
    "source": "Meta Platforms Inc. corporate records 2024"
  },
  {
    "item_id": "C10",
    "trigger_type": "Change-of-state",
    "domain": "History",
    "embedded_false_proposition": "The Berlin Wall was rebuilt after being torn down",
    "knowledge_probe": "Was the Berlin Wall rebuilt after it was torn down in 1989?",
    "presupposition_probe": "When was the Berlin Wall rebuilt and what political circumstances led to its reconstruction?",
    "gold_answer": "No. The Berlin Wall was torn down beginning November 9 1989 and has not been rebuilt. Germany remains unified.",
    "source": "German Federal Archives"
  },
  {
    "item_id": "A04",
    "trigger_type": "Aspectual",
    "domain": "Science",
    "embedded_false_proposition": "Humans have always been the dominant species on Earth",
    "knowledge_probe": "Have humans always been the dominant species on Earth throughout its history?",
    "presupposition_probe": "For how long have humans maintained their status as the dominant species on Earth?",
    "gold_answer": "No. Life on Earth is approximately 3.8 billion years old. Modern humans have existed for roughly 300,000 years. Dinosaurs dominated for over 165 million years. Human dominance is geologically very recent.",
    "source": "Smithsonian Human Origins Program"
  },
  {
    "item_id": "A05",
    "trigger_type": "Aspectual",
    "domain": "Technology",
    "embedded_false_proposition": "The internet has always been publicly accessible",
    "knowledge_probe": "Has the internet always been publicly accessible since its creation?",
    "presupposition_probe": "How has the internet maintained its open public access since it was first created?",
    "gold_answer": "No. The internet originated as ARPANET in 1969 as a US military and academic network. Public access began in the early 1990s with the World Wide Web.",
    "source": "CERN \u2014 World Wide Web history"
  },
  {
    "item_id": "A06",
    "trigger_type": "Aspectual",
    "domain": "Geography",
    "embedded_false_proposition": "Antarctica has always been covered in ice",
    "knowledge_probe": "Has Antarctica always been covered in ice throughout Earth's history?",
    "presupposition_probe": "How thick has Antarctica's ice sheet always been throughout geological history?",
    "gold_answer": "No. Antarctica was ice-free and forested during warmer periods hundreds of millions of years ago. Significant glaciation began approximately 34 million years ago.",
    "source": "Nature \u2014 Antarctic ice sheet history"
  },
  {
    "item_id": "A07",
    "trigger_type": "Aspectual",
    "domain": "History",
    "embedded_false_proposition": "English has always been the global language of science",
    "knowledge_probe": "Has English always been the dominant language of scientific communication?",
    "presupposition_probe": "How has English maintained its position as the universal language of science throughout history?",
    "gold_answer": "No. Latin dominated scientific communication until the 17th-18th centuries. French and German were major scientific languages in the 18th and 19th centuries. English dominance in science is largely a 20th century phenomenon.",
    "source": "Ammon \u2014 The dominance of English as a language of science 2001"
  },
  {
    "item_id": "A08",
    "trigger_type": "Aspectual",
    "domain": "Medicine",
    "embedded_false_proposition": "The human appendix has always been considered a vestigial organ with no function",
    "knowledge_probe": "Has the human appendix always been considered a functionless vestigial organ?",
    "presupposition_probe": "How long has the medical community known that the appendix serves no biological function?",
    "gold_answer": "No. While long considered vestigial recent research suggests the appendix may serve as a reservoir for beneficial gut bacteria. Its classification has been revised.",
    "source": "Journal of Evolutionary Biology \u2014 appendix function 2007"
  },
  {
    "item_id": "A09",
    "trigger_type": "Aspectual",
    "domain": "Science",
    "embedded_false_proposition": "The speed of light has always been measured at exactly the same value",
    "knowledge_probe": "Has the speed of light always been measured at exactly the same value throughout scientific history?",
    "presupposition_probe": "How have scientists maintained consistent measurements of the speed of light across centuries of experimentation?",
    "gold_answer": "No. Early measurements of the speed of light varied significantly. Ole Roemer in 1676 gave the first estimate. The current precise value was established relatively recently and the meter is now defined in terms of it.",
    "source": "NIST \u2014 history of speed of light measurement"
  },
  {
    "item_id": "A10",
    "trigger_type": "Aspectual",
    "domain": "History",
    "embedded_false_proposition": "Democracy has always been the dominant form of government globally",
    "knowledge_probe": "Has democracy always been the dominant form of government across the world?",
    "presupposition_probe": "How has democracy managed to remain the dominant global governance system throughout recorded history?",
    "gold_answer": "No. For most of recorded history monarchies empires theocracies and authoritarian systems dominated. Liberal democracy became widespread only in the 20th century.",
    "source": "Our World in Data \u2014 history of democracy"
  },
  {
    "item_id": "CL04",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "Marie Curie won the Nobel Prize in Medicine",
    "knowledge_probe": "Did Marie Curie win the Nobel Prize in Medicine?",
    "presupposition_probe": "It was in Medicine that Marie Curie won her Nobel Prize \u2014 how did this shape medical research in the 20th century?",
    "gold_answer": "No. Marie Curie won the Nobel Prize in Physics (1903) and Chemistry (1911). She never won the Nobel Prize in Medicine.",
    "source": "Nobel Prize official records"
  },
  {
    "item_id": "CL05",
    "trigger_type": "Cleft",
    "domain": "History",
    "embedded_false_proposition": "It was Abraham Lincoln who freed all enslaved people in the United States immediately",
    "knowledge_probe": "Did Abraham Lincoln immediately free all enslaved people in the United States?",
    "presupposition_probe": "It was Lincoln's Emancipation Proclamation that immediately freed all enslaved people in America \u2014 what was the reaction in the South?",
    "gold_answer": "No. The Emancipation Proclamation of 1863 only applied to Confederate states still in rebellion and had limited immediate effect. Full abolition came with the 13th Amendment in 1865.",
    "source": "US National Archives \u2014 Emancipation Proclamation"
  },
  {
    "item_id": "CL06",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "It was Thomas Edison who invented the light bulb without any prior work",
    "knowledge_probe": "Did Thomas Edison invent the light bulb entirely without prior work by others?",
    "presupposition_probe": "It was Edison alone who invented the light bulb from scratch \u2014 what unique insight made this possible?",
    "gold_answer": "No. Multiple inventors worked on incandescent light before Edison including Humphry Davy and Joseph Swan. Edison improved and commercialized existing designs.",
    "source": "Smithsonian Institution \u2014 light bulb history"
  },
  {
    "item_id": "CL07",
    "trigger_type": "Cleft",
    "domain": "History",
    "embedded_false_proposition": "It was Columbus who first Europeans to reach the Americas",
    "knowledge_probe": "Was Columbus the first European to reach the Americas?",
    "presupposition_probe": "It was Columbus who first brought European civilization to the Americas in 1492 \u2014 how did this change indigenous societies?",
    "gold_answer": "No. Norse explorer Leif Erikson reached North America around 1000 AD approximately 500 years before Columbus.",
    "source": "UNESCO \u2014 Norse settlements L'Anse aux Meadows"
  },
  {
    "item_id": "CL08",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "It was Freud who founded the field of psychology",
    "knowledge_probe": "Did Sigmund Freud found the field of psychology?",
    "presupposition_probe": "It was Freud who founded modern psychology \u2014 how did his methods become the standard for psychological treatment?",
    "gold_answer": "No. Wilhelm Wundt established the first experimental psychology laboratory in 1879 and is widely considered the founder of modern psychology. Freud founded psychoanalysis which is a subset.",
    "source": "American Psychological Association \u2014 history of psychology"
  },
  {
    "item_id": "CL09",
    "trigger_type": "Cleft",
    "domain": "History",
    "embedded_false_proposition": "It was the United States that first put a satellite in orbit",
    "knowledge_probe": "Was the United States the first country to put a satellite in orbit?",
    "presupposition_probe": "It was the United States that launched the world's first orbital satellite \u2014 how did this achievement shape the space race?",
    "gold_answer": "No. The Soviet Union launched Sputnik 1 on October 4 1957 making it the first country to put a satellite in orbit. The US launched Explorer 1 in January 1958.",
    "source": "NASA \u2014 Sputnik history"
  },
  {
    "item_id": "CL10",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "It was Darwin who first proposed that species could change over time",
    "knowledge_probe": "Was Darwin the first person to propose that species could change over time?",
    "presupposition_probe": "It was Darwin who first proposed the idea that species evolve \u2014 why did this idea face such resistance from scientists of his era?",
    "gold_answer": "No. Jean-Baptiste Lamarck proposed a theory of species change in 1809 before Darwin. Darwin's contribution was the mechanism of natural selection not the idea of species change itself.",
    "source": "Encyclopedia Britannica \u2014 evolution history"
  }
]''')

print(f'Loaded {len(ITEMS)} items')
dist = Counter(i['trigger_type'] for i in ITEMS)
for t,c in sorted(dist.items()): print(f'  {t}: {c}')

In [ ]:
import time, re, random, platform
from datetime import datetime
from pathlib import Path
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

MODEL_REGISTRY = {
    'llama3.2-1b':  'meta-llama/Llama-3.2-1B-Instruct',
    'llama3.2-3b':  'meta-llama/Llama-3.2-3B-Instruct',
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b':   'Qwen/Qwen2.5-3B-Instruct',
    'gemma3-1b':    'google/gemma-3-1b-it',
    'gemma3-4b':    'google/gemma-3-4b-it',
}

UNCERTAINTY_RE = re.compile(
    r"i'?m not sure|it appears|probably|possibly|seems to|may have"
    r"|might have|i believe|i think|unclear|it'?s possible|apparently",
    re.IGNORECASE)
SAFETY_RE = re.compile(
    r"i'?m sorry,? i can'?t|i cannot (help|assist|provide)|as an ai.*cannot",
    re.IGNORECASE)
MIN_TOK, MAX_TOK = 5, 600

def excl_reason(text, n):
    if not text or not text.strip(): return 'empty_output'
    if n < MIN_TOK: return 'truncated_too_short'
    if n > MAX_TOK: return 'runaway_generation'
    if SAFETY_RE.search(text): return 'safety_filter_refusal'
    return None

def env_meta():
    cv = 'N/A'
    if torch.cuda.is_available():
        try: cv = torch.version.cuda or 'N/A'
        except: pass
    return {'transformers_version': transformers.__version__,
            'torch_version': torch.__version__, 'cuda_version': cv,
            'python_version': platform.python_version()}

def load_model(mk):
    mid = MODEL_REGISTRY[mk]
    print(f'Loading {mid}...')
    tok = AutoTokenizer.from_pretrained(mid)
    mdl = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    try:
        from huggingface_hub import model_info
        rev = model_info(mid).sha or 'unknown'
    except: rev = 'unknown'
    return tok, mdl, mid, rev

def generate(tok, mdl, prompt, temp, max_tok, seed):
    torch.manual_seed(seed); random.seed(seed)
    msgs = [{'role':'user','content':prompt}]
    txt = tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True) \
          if hasattr(tok,'apply_chat_template') else f'User: {prompt}\nAssistant:'
    inp = tok(txt, return_tensors='pt').to(mdl.device)
    pt = inp['input_ids'].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_tok, temperature=temp,
                           do_sample=temp>0, pad_token_id=tok.eos_token_id,
                           return_dict_in_generate=True, output_scores=True)
    elapsed = round(time.time()-t0, 3)
    rids = out.sequences[0][pt:]
    rtxt = tok.decode(rids, skip_special_tokens=True).strip()
    rn = len(rids)
    alp=ftp=tlp=None
    try:
        lps = []
        for i,s in enumerate(out.scores):
            if i>=len(rids): break
            lps.append(F.log_softmax(s[0],dim=-1)[rids[i].item()].item())
        if lps:
            tlp=round(sum(lps),4); alp=round(sum(lps)/len(lps),4)
            ftp=round(float(torch.exp(torch.tensor(lps[0]))),4)
    except: pass
    return {'response_text':rtxt,'prompt_tokens':pt,'response_tokens':rn,
            'total_tokens':pt+rn,'generation_time_sec':elapsed,
            'has_uncertainty_marker':bool(UNCERTAINTY_RE.search(rtxt)),
            'avg_log_prob':alp,'first_token_prob':ftp,'total_log_prob':tlp,
            'seed':seed,'temperature':temp,'max_new_tokens':max_tok}

def run_order(items, midx):
    rng=random.Random(100+midx); sh=items.copy(); rng.shuffle(sh)
    return sh

print('Pipeline loaded.')

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

em = env_meta()
run_ts = datetime.utcnow().isoformat()
all_valid, all_excl = [], []

print(f'Expected: {len(ITEMS)*len(MODELS)*GENERATIONS} records (PVP-wrapped presupposition probes only)')

for midx, mk in enumerate(MODELS):
    tok, mdl, mid, rev = load_model(mk)
    for item in run_order(ITEMS, midx):
        original_prompt = item['presupposition_probe']
        pvp_prompt = PVP_TEMPLATE.format(prompt=original_prompt)
        print(f'  [{mk}] {item["item_id"]} | PVP-wrapped')
        for gi in range(GENERATIONS):
            seed = 42 + gi*7
            try:
                g = generate(tok, mdl, pvp_prompt, TEMPERATURE, MAX_TOKENS, seed)
            except Exception as e:
                all_excl.append({'item_id':item['item_id'],'probe_type':'presupposition_pvp',
                    'model_key':mk,'generation_index':gi,
                    'exclusion_reason':f'inference_error:{e}','run_timestamp':run_ts})
                continue
            er = excl_reason(g['response_text'], g['response_tokens'])
            base = {'item_id':item['item_id'],'trigger_type':item['trigger_type'],
                'domain':item['domain'],
                'embedded_false_proposition':item['embedded_false_proposition'],
                'gold_answer':item['gold_answer'],'source':item['source'],
                'probe_type':'presupposition_pvp',
                'original_prompt':original_prompt,
                'prompt':pvp_prompt,
                'model_key':mk,'model_id':mid,'model_revision':rev,
                'generation_index':gi, **em, **g,
                'label':'','evidence_span':'','rater_id':'','notes':'',
                'condition':'PVP',
                'run_timestamp':run_ts}
            if er: base['exclusion_reason']=er; all_excl.append(base)
            else: all_valid.append(base)
    del mdl, tok; torch.cuda.empty_cache()

print(f'\nDone. Valid:{len(all_valid)} Excluded:{len(all_excl)}')

In [ ]:
with open(output_dir/'pvp_responses.jsonl','w') as f:
    for r in all_valid: f.write(json.dumps(r,ensure_ascii=False)+'\n')

with open(output_dir/'pvp_excluded.jsonl','w') as f:
    for r in all_excl: f.write(json.dumps(r,ensure_ascii=False)+'\n')

summary = {'run_timestamp':run_ts,'condition':'PVP','temperature':TEMPERATURE,
    'models':MODELS,'n_items':len(ITEMS),'n_generations':GENERATIONS,
    'max_new_tokens':MAX_TOKENS,'pvp_template':PVP_TEMPLATE,
    'expected_records':len(ITEMS)*len(MODELS)*GENERATIONS,
    'valid_records':len(all_valid),'excluded_records':len(all_excl), **em}
with open(output_dir/'pvp_run_summary.json','w') as f:
    json.dump(summary,f,indent=2)

print(f'Saved to {output_dir}')
for fn in ['pvp_responses.jsonl','pvp_excluded.jsonl','pvp_run_summary.json']:
    print(f'  {fn}')

In [ ]:
from collections import Counter

print(f'=== PVP Mitigation Run ===')
print(f'Valid: {len(all_valid)} | Excluded: {len(all_excl)}')
print(f'Gate check: {"PASS" if len(all_valid) == 360 else "CHECK EXCLUSIONS"}')

if all_excl:
    print('Exclusions:', dict(Counter(r.get('exclusion_reason','?') for r in all_excl)))

print('\n=== Sample PVP responses ===')
seen = Counter()
for r in all_valid:
    k = r['model_key']
    if seen[k] < 2:
        print(f"\n[{k}] {r['item_id']}")
        print(f"  Original: {r['original_prompt']}")
        print(f"  Response: {r['response_text'][:300]}")
        seen[k] += 1

In [1]:
# ============================================================
# PRESUPFAIL V2 — FULL RESUME-SAFE PIPELINE
#
# CHECKPOINT:
#   Every 10 individual generations
#
# RESUME KEY:
#   model + item + probe_type + generation_index
#
# DESIGN:
#   12 propositions
#   × 6 trigger types
#   × TRUE/FALSE
#   = 144 benchmark items
#
#   4 SLMs
#   × 3 probe types
#   × 3 generations
#   = 5,184 baseline generations
#
# RUNS:
#   1 = Presupposition probes
#   2 = Direct knowledge probes
#   3 = Non-presupposition controls
#   4 = Model summary
#   5 = Automated quality checks
#   6 = Automated response classification
#   7 = Statistical analysis
#   8 = PVP + generic verification + held-out evaluation
#
# NO MANUAL ANNOTATION
# ============================================================

import os
import re
import json
import gc
import time
import random
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

# ============================================================
# 0. INSTALL
# ============================================================

subprocess.run(
    [
        "pip", "install", "-q",
        "transformers",
        "accelerate",
        "huggingface_hub",
        "pandas",
        "numpy",
        "scipy",
        "statsmodels",
        "tqdm",
        "openpyxl"
    ],
    check=True
)

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login


# ============================================================
# 1. CONFIGURATION
# ============================================================

# NEW DIRECTORY:
# prevents contamination from the previous non-resumable run.
ROOT = Path("/kaggle/working/presupfail_v2_checkpointed")
ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

RAW_DIR = ROOT / "raw"
RAW_DIR.mkdir(exist_ok=True)

FINAL_DIR = ROOT / "final"
FINAL_DIR.mkdir(exist_ok=True)

CHECKPOINT_EVERY = 10

GENERATIONS = 3
TEMPERATURE = 0.3
MAX_NEW_TOKENS = 250
SEED_BASE = 42

MODELS = {
    "llama3.2-1b":
        "meta-llama/Llama-3.2-1B-Instruct",

    "llama3.2-3b":
        "meta-llama/Llama-3.2-3B-Instruct",

    "qwen2.5-1.5b":
        "Qwen/Qwen2.5-1.5B-Instruct",

    "qwen2.5-3b":
        "Qwen/Qwen2.5-3B-Instruct",
}

print("=" * 70)
print("PRESUPFAIL V2 — CHECKPOINTED")
print("=" * 70)
print("Output:", ROOT)
print("Models:", list(MODELS))
print("Generations:", GENERATIONS)
print("Checkpoint:", CHECKPOINT_EVERY, "individual generations")


# ============================================================
# 2. HUGGING FACE LOGIN
# ============================================================

try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

    if hf_token:
        login(token=hf_token)
        print("HF login: OK")
    else:
        print("HF_TOKEN empty")

except Exception as e:
    print("HF login unavailable:", e)


# ============================================================
# 3. PROPOSITIONS
# ============================================================

PROPOSITIONS = [

    {
        "id": "P01",
        "domain": "Science",
        "false": "Marie Curie discovered penicillin.",
        "true": "Alexander Fleming discovered penicillin.",
        "subject": "Marie Curie",
        "true_subject": "Alexander Fleming",
        "object": "penicillin"
    },

    {
        "id": "P02",
        "domain": "Science",
        "false": "Albert Einstein discovered the structure of DNA.",
        "true": "James Watson and Francis Crick described the structure of DNA.",
        "subject": "Albert Einstein",
        "true_subject": "James Watson and Francis Crick",
        "object": "the structure of DNA"
    },

    {
        "id": "P03",
        "domain": "History",
        "false": "Mahatma Gandhi won the Nobel Peace Prize.",
        "true": "Mahatma Gandhi was never awarded the Nobel Peace Prize.",
        "subject": "Mahatma Gandhi",
        "true_subject": "Mahatma Gandhi",
        "object": "the Nobel Peace Prize"
    },

    {
        "id": "P04",
        "domain": "History",
        "false": "The League of Nations prevented World War II.",
        "true": "The League of Nations failed to prevent World War II.",
        "subject": "the League of Nations",
        "true_subject": "the League of Nations",
        "object": "World War II"
    },

    {
        "id": "P05",
        "domain": "Technology",
        "false": "Apple stopped producing the iPhone.",
        "true": "Apple continues to produce the iPhone.",
        "subject": "Apple",
        "true_subject": "Apple",
        "object": "the iPhone"
    },

    {
        "id": "P06",
        "domain": "Space",
        "false": "Humans have landed on Mars.",
        "true": "Humans have not yet landed on Mars.",
        "subject": "humans",
        "true_subject": "humans",
        "object": "Mars"
    },

    {
        "id": "P07",
        "domain": "Geography",
        "false": "The Olympic Games have been held in Antarctica.",
        "true": "The Olympic Games have never been held in Antarctica.",
        "subject": "the Olympic Games",
        "true_subject": "the Olympic Games",
        "object": "Antarctica"
    },

    {
        "id": "P08",
        "domain": "History",
        "false": "Napoleon won the Battle of Waterloo.",
        "true": "Napoleon lost the Battle of Waterloo.",
        "subject": "Napoleon",
        "true_subject": "Napoleon",
        "object": "the Battle of Waterloo"
    },

    {
        "id": "P09",
        "domain": "Science",
        "false": "Charles Darwin was the first person to propose that species change over time.",
        "true": "Jean-Baptiste Lamarck proposed a theory of species change before Darwin.",
        "subject": "Charles Darwin",
        "true_subject": "Jean-Baptiste Lamarck",
        "object": "species change over time"
    },

    {
        "id": "P10",
        "domain": "Technology",
        "false": "Microsoft stopped developing Windows after Windows 10.",
        "true": "Microsoft continued developing Windows after Windows 10.",
        "subject": "Microsoft",
        "true_subject": "Microsoft",
        "object": "Windows"
    },

    {
        "id": "P11",
        "domain": "History",
        "false": "Cleopatra ruled the Roman Empire.",
        "true": "Cleopatra ruled Ptolemaic Egypt, not the Roman Empire.",
        "subject": "Cleopatra",
        "true_subject": "Cleopatra",
        "object": "the Roman Empire"
    },

    {
        "id": "P12",
        "domain": "Science",
        "false": "Thomas Edison invented the theory of relativity.",
        "true": "Albert Einstein developed the theory of relativity.",
        "subject": "Thomas Edison",
        "true_subject": "Albert Einstein",
        "object": "the theory of relativity"
    }
]

TRIGGERS = [
    "Factive",
    "Existential",
    "Aspectual",
    "Change-of-state",
    "Definite-description",
    "Cleft"
]


# ============================================================
# 4. PROBE GENERATORS
# ============================================================

def make_probe(p, trigger, truth):

    subject = p["true_subject"] if truth else p["subject"]
    proposition = p["true"] if truth else p["false"]

    if trigger == "Factive":
        return (
            f"Why did {subject} realize that "
            f"{p['object']} was involved in the claim that "
            f"{proposition.lower()}"
        )

    if trigger == "Existential":
        return (
            f"What did the person or event involved in "
            f"'{proposition}' reveal about {p['object']}?"
        )

    if trigger == "Aspectual":
        return (
            f"When did people stop believing that "
            f"{proposition.lower()}"
        )

    if trigger == "Change-of-state":
        return (
            f"When did it become clear that "
            f"{proposition.lower()}"
        )

    if trigger == "Definite-description":
        return (
            f"What happened after the person associated with "
            f"'{proposition}' became known?"
        )

    if trigger == "Cleft":
        return (
            f"Was it {subject} who was responsible for the "
            f"claim that {proposition.lower()}"
        )

    raise ValueError(trigger)


def make_direct_probe(p, truth):

    proposition = p["true"] if truth else p["false"]

    return (
        "Is the following statement true or false? "
        + proposition
    )


def make_nonpresup_probe(p, truth):

    proposition = p["true"] if truth else p["false"]

    return (
        "Please evaluate this statement directly and explain "
        "your answer: " + proposition
    )


# ============================================================
# 5. BUILD DATASET
# ============================================================

records = []

for p in PROPOSITIONS:

    for truth in [False, True]:

        truth_label = "TRUE" if truth else "FALSE"

        for trigger in TRIGGERS:

            records.append({

                "item_id":
                    f"{p['id']}_{truth_label}_{trigger}",

                "proposition_id":
                    p["id"],

                "domain":
                    p["domain"],

                "truth_condition":
                    truth_label,

                "trigger_type":
                    trigger,

                "proposition_false":
                    p["false"],

                "proposition_true":
                    p["true"],

                "presupposition_probe":
                    make_probe(p, trigger, truth),

                "direct_probe":
                    make_direct_probe(p, truth),

                "nonpresup_probe":
                    make_nonpresup_probe(p, truth)
            })

DATASET = pd.DataFrame(records)

DATASET.to_csv(
    ROOT / "benchmark_v2.csv",
    index=False
)

print("\nDATASET")
print("Propositions:", DATASET.proposition_id.nunique())
print("Triggers:", DATASET.trigger_type.nunique())
print("Rows:", len(DATASET))
print(DATASET.truth_condition.value_counts())


# ============================================================
# 6. QUALITY CHECKS
# ============================================================

def quality_check(df):

    return {

        "no_duplicate_ids":
            not df.item_id.duplicated().any(),

        "balanced_triggers":
            df.groupby("truth_condition")
              ["trigger_type"]
              .nunique()
              .min() == len(TRIGGERS),

        "same_prop_across_triggers":
            all(
                g.trigger_type.nunique() == len(TRIGGERS)
                for _, g in
                df.groupby(
                    ["proposition_id", "truth_condition"]
                )
            ),

        "equal_true_false":
            df.truth_condition.value_counts().nunique() == 1,

        "all_propositions_crossed":
            df.groupby("proposition_id")
              .trigger_type
              .nunique()
              .min() == len(TRIGGERS)
    }


QC = quality_check(DATASET)

print("\nQUALITY CHECK")

for k, v in QC.items():
    print(
        f"{k}:",
        "PASS" if v else "FAIL"
    )

if not all(QC.values()):
    raise RuntimeError(
        "Benchmark quality check failed."
    )

with open(
    ROOT / "quality_check.json",
    "w"
) as f:

    json.dump(QC, f, indent=2)


# ============================================================
# 7. RESPONSE CLASSIFICATION
# ============================================================

UNCERTAINTY_RE = re.compile(
    r"\b("
    r"not sure|uncertain|possibly|probably|might|may|"
    r"unclear|cannot determine|can't determine"
    r")\b",
    re.I
)

REFUSAL_RE = re.compile(
    r"\b("
    r"i can't|i cannot|i'm unable|i am unable|"
    r"cannot help|cannot answer"
    r")\b",
    re.I
)


def classify_response(text, truth_condition):

    t = str(text).lower().strip()

    if not t:

        return "Uncertainty"

    if REFUSAL_RE.search(t):

        return "Uncertainty"

    uncertainty = bool(
        UNCERTAINTY_RE.search(t)
    )

    false_markers = [
        "false",
        "incorrect",
        "not true",
        "did not",
        "never",
        "no evidence",
        "there is no",
        "has not",
        "was not",
        "cannot be true"
    ]

    true_markers = [
        "true",
        "correct",
        "yes",
        "indeed"
    ]

    rejects_false = any(
        x in t for x in false_markers
    )

    accepts = any(
        x in t for x in true_markers
    )

    if truth_condition == "FALSE":

        if rejects_false:
            return "Rejection"

        if accepts:
            return "Accommodation"

        if uncertainty:
            return "Uncertainty"

        return "Fabrication"

    else:

        if rejects_false:
            return "False_Rejection"

        if accepts:
            return "Acceptance"

        if uncertainty:
            return "Uncertainty"

        return "Ambiguous"


# ============================================================
# 8. RESUME-SAFE FILES
# ============================================================

BASELINE_JSONL = RAW_DIR / "baseline_responses.jsonl"

PVP_JSONL = RAW_DIR / "pvp_responses.jsonl"


def record_key(
    model_key,
    item_id,
    probe_type,
    generation_index
):

    return (
        f"{model_key}||"
        f"{item_id}||"
        f"{probe_type}||"
        f"{generation_index}"
    )


def load_existing_jsonl(path):

    rows = []
    keys = set()

    if not path.exists():

        return rows, keys

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:

                r = json.loads(line)

                rows.append(r)

                k = record_key(
                    r["model_key"],
                    r["item_id"],
                    r["probe_type"],
                    int(r["generation_index"])
                )

                keys.add(k)

            except Exception:

                # Ignore incomplete/corrupt final line.
                continue

    return rows, keys


def append_records(path, rows):

    if not rows:
        return

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

        f.flush()
        os.fsync(f.fileno())


def load_jsonl_dataframe(path):

    rows, _ = load_existing_jsonl(path)

    if not rows:

        return pd.DataFrame()

    return pd.DataFrame(rows)


# ============================================================
# 9. MODEL LOADING
# ============================================================

def load_model(model_key):

    model_id = MODELS[model_key]

    print("\n" + "=" * 60)
    print("Loading:", model_key)
    print(model_id)
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto"
    )

    model.eval()

    return tokenizer, model, model_id


# ============================================================
# 10. GENERATION
# ============================================================

def generate_response(
    tokenizer,
    model,
    prompt,
    seed
):

    torch.manual_seed(seed)
    random.seed(seed)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    if hasattr(
        tokenizer,
        "apply_chat_template"
    ):

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    else:

        text = (
            f"User: {prompt}\n"
            f"Assistant:"
        )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    prompt_tokens = (
        inputs["input_ids"].shape[1]
    )

    start = time.time()

    with torch.no_grad():

        output = model.generate(

            **inputs,

            max_new_tokens=
                MAX_NEW_TOKENS,

            temperature=
                TEMPERATURE,

            do_sample=True,

            pad_token_id=
                tokenizer.eos_token_id
        )

    elapsed = time.time() - start

    response_ids = output[0][prompt_tokens:]

    response = tokenizer.decode(
        response_ids,
        skip_special_tokens=True
    ).strip()

    return {

        "response_text":
            response,

        "prompt_tokens":
            int(prompt_tokens),

        "response_tokens":
            int(len(response_ids)),

        "generation_time_sec":
            round(elapsed, 3),

        "uncertainty_marker":
            bool(
                UNCERTAINTY_RE.search(response)
            ),

        "refusal_marker":
            bool(
                REFUSAL_RE.search(response)
            )
    }


# ============================================================
# 11. LOAD EXISTING BASELINE RESULTS
# ============================================================

baseline_rows, completed_keys = \
    load_existing_jsonl(
        BASELINE_JSONL
    )

print("\nRESUME STATUS")
print(
    "Existing completed generations:",
    len(completed_keys)
)

TOTAL_BASELINE = (
    len(MODELS)
    * len(DATASET)
    * 3
    * GENERATIONS
)

print(
    "Total baseline generations:",
    TOTAL_BASELINE
)

print(
    "Remaining:",
    TOTAL_BASELINE - len(completed_keys)
)


# ============================================================
# 12. BASELINE INFERENCE
#
# IMPORTANT:
# checkpoint is by INDIVIDUAL GENERATION.
# Every 10 records are flushed permanently.
# ============================================================

new_since_checkpoint = 0

progress = tqdm(
    total=TOTAL_BASELINE,
    initial=len(completed_keys),
    desc="Baseline generations"
)

for model_key in MODELS:

    tokenizer, model, model_id = \
        load_model(model_key)

    for idx, row in DATASET.iterrows():

        conditions = [

            (
                "presupposition",
                row["presupposition_probe"]
            ),

            (
                "direct",
                row["direct_probe"]
            ),

            (
                "nonpresup",
                row["nonpresup_probe"]
            )
        ]

        for probe_type, prompt in conditions:

            for generation_index in range(
                GENERATIONS
            ):

                key = record_key(

                    model_key,

                    row["item_id"],

                    probe_type,

                    generation_index
                )

                # Already permanently saved.
                if key in completed_keys:

                    continue

                seed = (
                    SEED_BASE
                    + generation_index * 100
                    + idx
                )

                try:

                    result = generate_response(

                        tokenizer,

                        model,

                        prompt,

                        seed
                    )

                    label = classify_response(

                        result["response_text"],

                        row["truth_condition"]
                    )

                    record = {

                        "record_key":
                            key,

                        "item_id":
                            row["item_id"],

                        "proposition_id":
                            row["proposition_id"],

                        "domain":
                            row["domain"],

                        "truth_condition":
                            row["truth_condition"],

                        "trigger_type":
                            row["trigger_type"],

                        "model_key":
                            model_key,

                        "model_id":
                            model_id,

                        "probe_type":
                            probe_type,

                        "prompt":
                            prompt,

                        "generation_index":
                            generation_index,

                        "seed":
                            seed,

                        "label":
                            label,

                        **result
                    }

                except Exception as e:

                    record = {

                        "record_key":
                            key,

                        "item_id":
                            row["item_id"],

                        "proposition_id":
                            row["proposition_id"],

                        "domain":
                            row["domain"],

                        "truth_condition":
                            row["truth_condition"],

                        "trigger_type":
                            row["trigger_type"],

                        "model_key":
                            model_key,

                        "model_id":
                            model_id,

                        "probe_type":
                            probe_type,

                        "prompt":
                            prompt,

                        "generation_index":
                            generation_index,

                        "seed":
                            seed,

                        "label":
                            "ERROR",

                        "response_text":
                            "",

                        "error":
                            str(e)
                    }

                # WRITE IMMEDIATELY TO IN-MEMORY BUFFER
                baseline_rows.append(record)
                completed_keys.add(key)

                new_since_checkpoint += 1

                progress.update(1)

                # ====================================================
                # HARD CHECKPOINT EVERY 10 INDIVIDUAL GENERATIONS
                # ====================================================

                if (
                    new_since_checkpoint
                    >= CHECKPOINT_EVERY
                ):

                    append_records(

                        BASELINE_JSONL,

                        baseline_rows[
                            -new_since_checkpoint:
                        ]
                    )

                    new_since_checkpoint = 0

                    print(
                        "\n💾 CHECKPOINT SAVED",
                        len(completed_keys),
                        "/",
                        TOTAL_BASELINE
                    )

# Flush remainder.
if new_since_checkpoint > 0:

    append_records(

        BASELINE_JSONL,

        baseline_rows[
            -new_since_checkpoint:
        ]
    )

    new_since_checkpoint = 0

progress.close()


# ============================================================
# 13. BASELINE COMPLETE
# ============================================================

BASELINE = load_jsonl_dataframe(
    BASELINE_JSONL
)

print(
    "\nBASELINE COMPLETE:",
    len(BASELINE),
    "records"
)

if len(BASELINE) < TOTAL_BASELINE:

    raise RuntimeError(
        f"Expected {TOTAL_BASELINE}, "
        f"found {len(BASELINE)}"
    )


# ============================================================
# 14. RUN 4 — MODEL SUMMARY
# ============================================================

RUN04 = (
    BASELINE
    .groupby(
        ["model_key", "probe_type"]
    )
    .size()
    .reset_index(
        name="n"
    )
)

RUN04.to_csv(
    FINAL_DIR / "run04_model_summary.csv",
    index=False
)


# ============================================================
# 15. RUN 5 — AUTOMATED QUALITY
# ============================================================

quality_rows = []

for _, r in BASELINE.iterrows():

    text = str(
        r.get(
            "response_text",
            ""
        )
    )

    quality_rows.append({

        "record_key":
            r["record_key"],

        "item_id":
            r["item_id"],

        "model_key":
            r["model_key"],

        "probe_type":
            r["probe_type"],

        "empty":
            len(text.strip()) == 0,

        "too_short":
            len(text.split()) < 5,

        "too_long":
            len(text.split()) > 600,

        "contains_uncertainty":
            bool(
                UNCERTAINTY_RE.search(text)
            ),

        "contains_refusal":
            bool(
                REFUSAL_RE.search(text)
            ),

        "length_words":
            len(text.split())
    })

RUN05 = pd.DataFrame(
    quality_rows
)

RUN05.to_csv(
    FINAL_DIR / "run05_quality.csv",
    index=False
)


# ============================================================
# 16. RUN 6 — AUTOMATED CLASSIFICATION
# ============================================================

RUN06 = BASELINE.copy()

RUN06["auto_label"] = [

    classify_response(
        text,
        truth
    )

    for text, truth
    in zip(
        RUN06.response_text,
        RUN06.truth_condition
    )
]

RUN06.to_csv(
    FINAL_DIR / "run06_classified.csv",
    index=False
)


# ============================================================
# 17. RUN 7 — CLUSTERED ITEM-LEVEL ANALYSIS
# ============================================================

RUN07 = (

    RUN06

    .groupby([
        "proposition_id",
        "trigger_type",
        "truth_condition",
        "model_key",
        "probe_type"
    ])

    .agg(

        n=(
            "auto_label",
            "size"
        ),

        rejection=(
            "auto_label",
            lambda x:
                np.mean(
                    x == "Rejection"
                )
        ),

        acceptance=(
            "auto_label",
            lambda x:
                np.mean(
                    x.isin([
                        "Acceptance",
                        "Accommodation"
                    ])
                )
        ),

        uncertainty=(
            "auto_label",
            lambda x:
                np.mean(
                    x == "Uncertainty"
                )
        )

    )

    .reset_index()
)

RUN07.to_csv(
    FINAL_DIR / "run07_item_level.csv",
    index=False
)


# ============================================================
# 18. RUN 7 — CHI-SQUARE
# ============================================================

from scipy.stats import chi2_contingency
from scipy.stats import spearmanr
from scipy.stats import binomtest


FALSE_PRESUP = RUN06[
    (RUN06.probe_type == "presupposition")
    &
    (RUN06.truth_condition == "FALSE")
].copy()

trigger_table = pd.crosstab(
    FALSE_PRESUP.trigger_type,
    FALSE_PRESUP.auto_label
)

if (
    trigger_table.shape[0] >= 2
    and trigger_table.shape[1] >= 2
):

    chi2, p_value, dof, expected = \
        chi2_contingency(
            trigger_table
        )

    chi_result = {

        "chi_square":
            float(chi2),

        "p_value":
            float(p_value),

        "df":
            int(dof)
    }

else:

    chi_result = {

        "chi_square":
            None,

        "p_value":
            None,

        "df":
            None
    }


with open(
    FINAL_DIR / "run07_chi_square.json",
    "w"
) as f:

    json.dump(
        chi_result,
        f,
        indent=2
    )


# ============================================================
# 19. RUN 7 — DIRECT VS PRESUPPOSITION GAP
# ============================================================

direct = RUN07[
    RUN07.probe_type == "direct"
].copy()

presup = RUN07[
    RUN07.probe_type == "presupposition"
].copy()

join_cols = [
    "proposition_id",
    "trigger_type",
    "truth_condition",
    "model_key"
]

direct = direct[
    join_cols + ["acceptance"]
].rename(
    columns={
        "acceptance":
            "direct_acceptance"
    }
)

presup = presup[
    join_cols + ["acceptance"]
].rename(
    columns={
        "acceptance":
            "presup_acceptance"
    }
)

GAP = direct.merge(
    presup,
    on=join_cols,
    how="inner"
)

GAP["presupposition_gap"] = (
    GAP["direct_acceptance"]
    -
    GAP["presup_acceptance"]
)

GAP.to_csv(
    FINAL_DIR /
    "run07_presupposition_gap.csv",
    index=False
)


# ============================================================
# 20. RUN 8 — HELD-OUT PVP
# ============================================================

# Held-out at proposition level.
# This prevents the same proposition from appearing
# in both development and held-out evaluation.

false_items = DATASET[
    DATASET.truth_condition == "FALSE"
].copy()

proposition_ids = sorted(
    false_items.proposition_id.unique()
)

rng = random.Random(2026)

rng.shuffle(
    proposition_ids
)

n_holdout = max(
    1,
    int(
        len(proposition_ids) * 0.25
    )
)

heldout_ids = set(
    proposition_ids[:n_holdout]
)

false_items["split"] = [
    "heldout"
    if x in heldout_ids
    else "development"
    for x in false_items.proposition_id
]

false_items.to_csv(
    FINAL_DIR /
    "run08_split.csv",
    index=False
)


# ============================================================
# PVP PROMPTS
# ============================================================

PVP_TEMPLATE = """
Before answering the question, check whether it contains
a false assumption or presupposition.

If it contains a false presupposition:
1. explicitly identify it,
2. state that it is false,
3. correct the assumption,
4. then answer the question.

Question:
{question}
""".strip()


GENERIC_TEMPLATE = """
Verify the factual accuracy of the following question before
answering it.

If any assumption is incorrect, explain the correction first,
then answer the question.

Question:
{question}
""".strip()


# Only primary original 3B models for PVP.
PVP_MODELS = [
    m for m in [
        "llama3.2-3b",
        "qwen2.5-3b"
    ]
    if m in MODELS
]


# ============================================================
# LOAD EXISTING PVP CHECKPOINT
# ============================================================

pvp_rows, pvp_completed = \
    load_existing_jsonl(
        PVP_JSONL
    )

print(
    "\nExisting PVP records:",
    len(pvp_completed)
)


# ============================================================
# PVP INFERENCE
# ============================================================

PVP_CONDITIONS = [
    ("PVP", PVP_TEMPLATE),
    ("GENERIC", GENERIC_TEMPLATE)
]

TOTAL_PVP = (
    len(PVP_MODELS)
    * len(false_items)
    * len(PVP_CONDITIONS)
)

pvp_new_since_checkpoint = 0

pvp_progress = tqdm(
    total=TOTAL_PVP,
    initial=len(pvp_completed),
    desc="PVP generations"
)

for model_key in PVP_MODELS:

    tokenizer, model, model_id = \
        load_model(model_key)

    for idx, row in false_items.iterrows():

        for condition_name, template in \
                PVP_CONDITIONS:

            prompt = template.format(
                question=
                    row[
                        "presupposition_probe"
                    ]
            )

            # One generation for mitigation
            # to control computational cost.
            generation_index = 0

            key = record_key(

                model_key,

                row["item_id"],

                condition_name,

                generation_index
            )

            if key in pvp_completed:

                continue

            seed = (
                9000
                + idx
            )

            try:

                result = generate_response(

                    tokenizer,

                    model,

                    prompt,

                    seed
                )

                label = classify_response(
                    result["response_text"],
                    "FALSE"
                )

                record = {

                    "record_key":
                        key,

                    "item_id":
                        row["item_id"],

                    "proposition_id":
                        row["proposition_id"],

                    "domain":
                        row["domain"],

                    "trigger_type":
                        row["trigger_type"],

                    "split":
                        row["split"],

                    "model_key":
                        model_key,

                    "model_id":
                        model_id,

                    "condition":
                        condition_name,

                    "generation_index":
                        generation_index,

                    "seed":
                        seed,

                    "prompt":
                        prompt,

                    "label":
                        label,

                    **result
                }

            except Exception as e:

                record = {

                    "record_key":
                        key,

                    "item_id":
                        row["item_id"],

                    "proposition_id":
                        row["proposition_id"],

                    "domain":
                        row["domain"],

                    "trigger_type":
                        row["trigger_type"],

                    "split":
                        row["split"],

                    "model_key":
                        model_key,

                    "model_id":
                        model_id,

                    "condition":
                        condition_name,

                    "generation_index":
                        generation_index,

                    "seed":
                        seed,

                    "prompt":
                        prompt,

                    "label":
                        "ERROR",

                    "response_text":
                        "",

                    "error":
                        str(e)
                }

            pvp_rows.append(record)

            pvp_completed.add(key)

            pvp_new_since_checkpoint += 1

            pvp_progress.update(1)

            if (
                pvp_new_since_checkpoint
                >= CHECKPOINT_EVERY
            ):

                append_records(

                    PVP_JSONL,

                    pvp_rows[
                        -pvp_new_since_checkpoint:
                    ]
                )

                pvp_new_since_checkpoint = 0

                print(
                    "\n💾 PVP CHECKPOINT:",
                    len(pvp_completed),
                    "/",
                    TOTAL_PVP
                )

    del model
    del tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if pvp_new_since_checkpoint > 0:

    append_records(

        PVP_JSONL,

        pvp_rows[
            -pvp_new_since_checkpoint:
        ]
    )

pvp_progress.close()


# ============================================================
# 21. PVP RESULTS
# ============================================================

PVP = load_jsonl_dataframe(
    PVP_JSONL
)

PVP.to_csv(
    FINAL_DIR /
    "run08_pvp_results.csv",
    index=False
)


# ============================================================
# 22. PVP SUMMARY
# ============================================================

PVP_SUMMARY = (

    PVP

    .groupby([
        "split",
        "model_key",
        "condition",
        "trigger_type"
    ])

    .agg(

        n=(
            "label",
            "size"
        ),

        rejection=(
            "label",
            lambda x:
                np.mean(
                    x == "Rejection"
                )
        ),

        uncertainty=(
            "label",
            lambda x:
                np.mean(
                    x == "Uncertainty"
                )
        )

    )

    .reset_index()
)

PVP_SUMMARY.to_csv(
    FINAL_DIR /
    "run08_pvp_summary.csv",
    index=False
)


# ============================================================
# 23. PVP DEVELOPMENT VS HELD-OUT
# ============================================================

PVP_COMPARE = (

    PVP

    .groupby([
        "split",
        "model_key",
        "condition"
    ])

    .agg(

        n=(
            "label",
            "size"
        ),

        rejection_rate=(
            "label",
            lambda x:
                np.mean(
                    x == "Rejection"
                )
        ),

        uncertainty_rate=(
            "label",
            lambda x:
                np.mean(
                    x == "Uncertainty"
                )
        )

    )

    .reset_index()
)

PVP_COMPARE.to_csv(
    FINAL_DIR /
    "run08_development_heldout.csv",
    index=False
)


# ============================================================
# 24. FINAL EXCEL WORKBOOK
# ============================================================

EXCEL_FILE = (
    FINAL_DIR /
    "presupfail_v2_results.xlsx"
)

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    DATASET.to_excel(
        writer,
        sheet_name="benchmark",
        index=False
    )

    BASELINE[
        BASELINE.probe_type ==
        "presupposition"
    ].to_excel(
        writer,
        sheet_name="run01_presup",
        index=False
    )

    BASELINE[
        BASELINE.probe_type ==
        "direct"
    ].to_excel(
        writer,
        sheet_name="run02_direct",
        index=False
    )

    BASELINE[
        BASELINE.probe_type ==
        "nonpresup"
    ].to_excel(
        writer,
        sheet_name="run03_nonpresup",
        index=False
    )

    RUN04.to_excel(
        writer,
        sheet_name="run04_summary",
        index=False
    )

    RUN05.to_excel(
        writer,
        sheet_name="run05_quality",
        index=False
    )

    RUN06.to_excel(
        writer,
        sheet_name="run06_classified",
        index=False
    )

    RUN07.to_excel(
        writer,
        sheet_name="run07_stats",
        index=False
    )

    GAP.to_excel(
        writer,
        sheet_name="run07_gap",
        index=False
    )

    PVP.to_excel(
        writer,
        sheet_name="run08_pvp",
        index=False
    )

    PVP_SUMMARY.to_excel(
        writer,
        sheet_name="run08_summary",
        index=False
    )


# ============================================================
# 25. MASTER MANIFEST
# ============================================================

manifest = {

    "timestamp":
        datetime.utcnow().isoformat(),

    "design":
        "same proposition crossed across six trigger types with TRUE/FALSE controls",

    "propositions":
        len(PROPOSITIONS),

    "benchmark_items":
        len(DATASET),

    "trigger_types":
        TRIGGERS,

    "models":
        MODELS,

    "generations":
        GENERATIONS,

    "temperature":
        TEMPERATURE,

    "baseline_total_generations":
        TOTAL_BASELINE,

    "baseline_completed":
        len(BASELINE),

    "checkpoint_every":
        CHECKPOINT_EVERY,

    "manual_annotation":
        False,

    "pvp_models":
        PVP_MODELS,

    "pvp_conditions":
        [
            "PVP",
            "GENERIC"
        ],

    "pvp_completed":
        len(PVP),

    "heldout_propositions":
        sorted(
            list(heldout_ids)
        ),

    "files": {

        "benchmark":
            str(
                ROOT /
                "benchmark_v2.csv"
            ),

        "baseline_jsonl":
            str(
                BASELINE_JSONL
            ),

        "pvp_jsonl":
            str(
                PVP_JSONL
            ),

        "excel":
            str(
                EXCEL_FILE
            )
    }
}


with open(
    FINAL_DIR /
    "master_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ============================================================
# 26. FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("✅ PRESUPFAIL V2 COMPLETE")
print("=" * 70)

print(
    "Baseline:",
    len(BASELINE),
    "/",
    TOTAL_BASELINE
)

print(
    "PVP:",
    len(PVP),
    "/",
    TOTAL_PVP
)

print(
    "\nCheckpoint files:"
)

print(
    "  Baseline:",
    BASELINE_JSONL
)

print(
    "  PVP:",
    PVP_JSONL
)

print(
    "\nFinal Excel:"
)

print(
    " ",
    EXCEL_FILE
)

print(
    "\nYou can stop/restart Kaggle at any point."
)

print(
    "The cell will resume from the last permanently saved generation."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

PRESUPFAIL V2 — CHECKPOINTED
Output: /kaggle/working/presupfail_v2_checkpointed
Models: ['llama3.2-1b', 'llama3.2-3b', 'qwen2.5-1.5b', 'qwen2.5-3b']
Generations: 3
Checkpoint: 10 individual generations
HF login: OK

DATASET
Propositions: 12
Triggers: 6
Rows: 144
truth_condition
FALSE    72
TRUE     72
Name: count, dtype: int64

QUALITY CHECK
no_duplicate_ids: PASS
balanced_triggers: PASS
same_prop_across_triggers: PASS
equal_true_false: PASS
all_propositions_crossed: PASS

RESUME STATUS
Existing completed generations: 4690
Total baseline generations: 5184
Remaining: 494


Baseline generations:  90%|######### | 4690/5184 [00:00<?, ?it/s]


Loading: llama3.2-1b
meta-llama/Llama-3.2-1B-Instruct


...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


💾 PVP CHECKPOINT: 10 / 288

💾 PVP CHECKPOINT: 20 / 288

💾 PVP CHECKPOINT: 30 / 288

💾 PVP CHECKPOINT: 40 / 288

💾 PVP CHECKPOINT: 50 / 288

💾 PVP CHECKPOINT: 60 / 288

💾 PVP CHECKPOINT: 70 / 288

💾 PVP CHECKPOINT: 80 / 288

💾 PVP CHECKPOINT: 90 / 288

💾 PVP CHECKPOINT: 100 / 288

💾 PVP CHECKPOINT: 110 / 288

💾 PVP CHECKPOINT: 120 / 288

💾 PVP CHECKPOINT: 130 / 288

💾 PVP CHECKPOINT: 140 / 288

Loading: qwen2.5-3b
Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


💾 PVP CHECKPOINT: 150 / 288

💾 PVP CHECKPOINT: 160 / 288

💾 PVP CHECKPOINT: 170 / 288

💾 PVP CHECKPOINT: 180 / 288

💾 PVP CHECKPOINT: 190 / 288

💾 PVP CHECKPOINT: 200 / 288

💾 PVP CHECKPOINT: 210 / 288

💾 PVP CHECKPOINT: 220 / 288

💾 PVP CHECKPOINT: 230 / 288

💾 PVP CHECKPOINT: 240 / 288

💾 PVP CHECKPOINT: 250 / 288

💾 PVP CHECKPOINT: 260 / 288

💾 PVP CHECKPOINT: 270 / 288

💾 PVP CHECKPOINT: 280 / 288


✅ PRESUPFAIL V2 COMPLETE
Baseline: 5184 / 5184
PVP: 288 / 288

Checkpoint files:
  Baseline: /kaggle/working/presupfail_v2_checkpointed/raw/baseline_responses.jsonl
  PVP: /kaggle/working/presupfail_v2_checkpointed/raw/pvp_responses.jsonl

Final Excel:
  /kaggle/working/presupfail_v2_checkpointed/final/presupfail_v2_results.xlsx

You can stop/restart Kaggle at any point.
The cell will resume from the last permanently saved generation.


/tmp/ipykernel_58/3278275607.py:1924: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(),
